In [1]:
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, roc_auc_score
import joblib
import numpy as np
import random
import os

random_seed = 42

def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # if using multi-GPU
    
    # Force deterministic operations
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(random_seed)

EMBEDDING_MODEL = "dinov3"

def load_and_concatenate_splits(file_paths):
    X_list, y_list = [], []
    for path in file_paths:
        data = torch.load(path, weights_only=True)
        X_list.append(data["features"])
        y_list.append(data["labels"])
    
    X_combined = torch.cat(X_list, dim=0)
    y_combined = torch.cat(y_list, dim=0).float()
    return X_combined, y_combined

X_train, y_train = load_and_concatenate_splits([
    f"extracted_embeddings/pitfall_train_embeddings_{EMBEDDING_MODEL}.pt",
    f"extracted_embeddings/ecostack_train_embeddings_{EMBEDDING_MODEL}.pt"
])
X_val, y_val = load_and_concatenate_splits([
    f"extracted_embeddings/pitfall_val_embeddings_{EMBEDDING_MODEL}.pt",
    f"extracted_embeddings/ecostack_val_embeddings_{EMBEDDING_MODEL}.pt"
])

X_test_idd , y_test_idd = load_and_concatenate_splits([
    f"extracted_embeddings/pitfall_test_embeddings_{EMBEDDING_MODEL}.pt",
    f"extracted_embeddings/ecostack_test_embeddings_{EMBEDDING_MODEL}.pt"
])

X_test_ood , y_test_ood = load_and_concatenate_splits([
    f"extracted_embeddings/rangex_test_embeddings_{EMBEDDING_MODEL}.pt",
    f"extracted_embeddings/stefan_test_embeddings_{EMBEDDING_MODEL}.pt"
])

print(f"Train size: {X_train.shape[0]} | Val size: {X_val.shape[0]} | Test(IDD) size: {X_test_idd.shape[0]} | Test(OOD) size: {X_test_ood.shape[0]}")

# because embeddings are almost linearly separable, also to reduce training time
base_svm = LinearSVC(
    class_weight="balanced", 
    random_state=random_seed, 
    max_iter=3000, 
    dual=False 
)

calibrated_svm = CalibratedClassifierCV(base_svm, cv=5, method='sigmoid')

svm_pipeline = make_pipeline(
    StandardScaler(),
    calibrated_svm
)

# CROSS VALIDATION
from sklearn.model_selection import cross_validate
cv_results = cross_validate(
    svm_pipeline, 
    X_train, 
    y_train, 
    cv=5, 
    scoring=['precision', 'recall', 'f1_macro', 'roc_auc'],
    n_jobs=-1 
)

precision_scores = cv_results['test_precision']
recall_scores = cv_results['test_recall']
f1_scores = cv_results['test_f1_macro']
roc_auc_scores = cv_results['test_roc_auc']

print(f"CV F1-score (Macro): {f1_scores.mean():.4f} +/- {f1_scores.std():.4f}")
print(f"CV Recall:           {recall_scores.mean():.4f} +/- {recall_scores.std():.4f}")
print(f"CV Precision:        {precision_scores.mean():.4f} +/- {precision_scores.std():.4f}")
print(f"CV ROC-AUC:          {roc_auc_scores.mean():.4f} +/- {roc_auc_scores.std():.4f}")

# -----------------------------------------------------------------------------------------------

svm_pipeline.fit(X_train, y_train)

def evaluate_sklearn_model(X, y, split_name):
    print(f"\n--- {split_name} Results ---")
    y_pred = svm_pipeline.predict(X)
    y_proba = svm_pipeline.predict_proba(X)[:, 1]
    
    print(classification_report(y, y_pred, target_names=["background", "insect"]))
    print(f"ROC-AUC: {roc_auc_score(y, y_proba):.4f}")

evaluate_sklearn_model(X_val, y_val, "Validation")
evaluate_sklearn_model(X_test_idd, y_test_idd, "Test (IDD)")
evaluate_sklearn_model(X_test_ood, y_test_ood, "Test (OOD)")

save_dir = os.path.join(os.environ.get("MODEL_CACHE_DIR", os.path.join("..", "models-cache")), "binary_classificator")
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f"insect_classifier_ecostack_svm_{EMBEDDING_MODEL}.joblib")
joblib.dump(svm_pipeline, save_path)
print(f"\nModel saved to {save_path}")

Train size: 59358 | Val size: 8938 | Test(IDD) size: 20200 | Test(OOD) size: 50566
CV F1-score (Macro): 0.9487 +/- 0.0685
CV Recall:           0.9227 +/- 0.1114
CV Precision:        0.9712 +/- 0.0321
CV ROC-AUC:          0.9747 +/- 0.0462

--- Validation Results ---
              precision    recall  f1-score   support

  background       0.97      0.98      0.97      4469
      insect       0.98      0.97      0.97      4469

    accuracy                           0.97      8938
   macro avg       0.97      0.97      0.97      8938
weighted avg       0.97      0.97      0.97      8938

ROC-AUC: 0.9963

--- Test (IDD) Results ---
              precision    recall  f1-score   support

  background       0.94      0.98      0.96     10100
      insect       0.98      0.94      0.96     10100

    accuracy                           0.96     20200
   macro avg       0.96      0.96      0.96     20200
weighted avg       0.96      0.96      0.96     20200

ROC-AUC: 0.9913

--- Test (OOD) Res

BioCLIP 2.5
- CV F1-score (Macro): 0.9225 +/- 0.1242
- CV Recall:           0.8699 +/- 0.2270
- CV Precision:        0.9816 +/- 0.0084
- CV ROC-AUC:          0.9775 +/- 0.0424

DINOv3
- CV F1-score (Macro): 0.9487 +/- 0.0685
- CV Recall:           0.9227 +/- 0.1114
- CV Precision:        0.9712 +/- 0.0321
- CV ROC-AUC:          0.9747 +/- 0.0462

In [ ]:
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, roc_auc_score
import joblib
import numpy as np
import random
import os

random_seed = 42

def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # if using multi-GPU
    
    # Force deterministic operations
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(random_seed)

EMBEDDING_MODEL = "dinov3"

def load_and_concatenate_splits(file_paths):
    X_list, y_list = [], []
    for path in file_paths:
        data = torch.load(path, weights_only=True)
        X_list.append(data["features"])
        y_list.append(data["labels"])
    
    X_combined = torch.cat(X_list, dim=0)
    y_combined = torch.cat(y_list, dim=0).float()
    return X_combined, y_combined

def evaluate_sklearn_model(X, y, split_name):
    print(f"\n--- {split_name} Results ---")
    y_pred = svm_pipeline.predict(X)
    y_proba = svm_pipeline.predict_proba(X)[:, 1]
    
    print(classification_report(y, y_pred, target_names=["background", "insect"]))
    print(f"ROC-AUC: {roc_auc_score(y, y_proba):.4f}")

X_train, y_train = load_and_concatenate_splits([
    f"extracted_embeddings/pitfall_train_embeddings_{EMBEDDING_MODEL}.pt",
    f"extracted_embeddings/ecostack_train_embeddings_{EMBEDDING_MODEL}.pt"
])
X_val, y_val = load_and_concatenate_splits([
    f"extracted_embeddings/pitfall_val_embeddings_{EMBEDDING_MODEL}.pt",
    f"extracted_embeddings/ecostack_val_embeddings_{EMBEDDING_MODEL}.pt"
])

X_test_idd , y_test_idd = load_and_concatenate_splits([
    f"extracted_embeddings/pitfall_test_embeddings_{EMBEDDING_MODEL}.pt",
    f"extracted_embeddings/ecostack_test_embeddings_{EMBEDDING_MODEL}.pt"
])

X_test_ood , y_test_ood = load_and_concatenate_splits([
    f"extracted_embeddings/rangex_test_embeddings_{EMBEDDING_MODEL}.pt",
    f"extracted_embeddings/stefan_test_embeddings_{EMBEDDING_MODEL}.pt"
])

print(f"Train size: {X_train.shape[0]} | Val size: {X_val.shape[0]} | Test(IDD) size: {X_test_idd.shape[0]} | Test(OOD) size: {X_test_ood.shape[0]}")

# because embeddings are almost linearly separable, also to reduce training time
base_svm = LinearSVC(
    class_weight="balanced", 
    random_state=random_seed, 
    max_iter=3000, 
    dual=False 
)

calibrated_svm = CalibratedClassifierCV(base_svm, cv=5, method='sigmoid')

svm_pipeline = make_pipeline(
    StandardScaler(),
    calibrated_svm
)

save_dir = os.path.join(os.environ.get("MODEL_CACHE_DIR", os.path.join("..", "models-cache")), "binary_classificator")
save_path = os.path.join(save_dir, f"insect_classifier_ecostack_svm_{EMBEDDING_MODEL}.joblib")

svm_pipeline = joblib.load(save_path)
evaluate_sklearn_model(X_test_idd, y_test_idd, "Test (IDD) - Loaded Model")
evaluate_sklearn_model(X_test_ood, y_test_ood, "Test (OOD) - Loaded Model")

Train size: 59358 | Val size: 8938 | Test(IDD) size: 20200 | Test(OOD) size: 50566

--- Test (IDD) - Loaded Model Results ---
              precision    recall  f1-score   support

  background       0.94      0.98      0.96     10100
      insect       0.98      0.94      0.96     10100

    accuracy                           0.96     20200
   macro avg       0.96      0.96      0.96     20200
weighted avg       0.96      0.96      0.96     20200

ROC-AUC: 0.9913

--- Test (OOD) - Loaded Model Results ---
              precision    recall  f1-score   support

  background       0.85      0.83      0.84     24133
      insect       0.85      0.86      0.86     26433

    accuracy                           0.85     50566
   macro avg       0.85      0.85      0.85     50566
weighted avg       0.85      0.85      0.85     50566

ROC-AUC: 0.9217
